# FashionMNIST Image Classifier

Run the cells below in order.

## 1. Load the data

Load FashionMNIST with TorchVision into a training/validation dataset and a
test dataset, then split the train/validation data randomly (seed 42) into
55,000 training and 5,000 validation samples.

In [ ]:
import torch
from torch.utils.data import random_split, DataLoader
import torchvision
from torchvision import transforms

# Convert the PIL images to tensors.
transform = transforms.ToTensor()

# Training/validation dataset (60,000 images).
train_val_dataset = torchvision.datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

# Test dataset (10,000 images).
test_dataset = torchvision.datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

# Split the 60,000 train/val images into 55,000 train and 5,000 validation,
# using a fixed seed of 42 for reproducibility.
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(
    train_val_dataset,
    [55_000, 5_000],
    generator=generator,
)

print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")

## 2. Create the DataLoaders

Wrap the datasets in DataLoaders with a batch size of 32. The training loader
is shuffled (seed 42 for a reproducible batch order); the validation and test
loaders are not shuffled.

In [ ]:
BATCH_SIZE = 32

# Seed the generator that drives the training loader's shuffling so the
# batch order is reproducible.
loader_generator = torch.Generator().manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Batch size: {BATCH_SIZE}")
print(f"Training batches:   {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

# Sanity check: inspect the shape of one training batch.
images, labels = next(iter(train_loader))
print(f"Image batch shape:  {tuple(images.shape)}")
print(f"Label batch shape:  {tuple(labels.shape)}")

## 3. Sample the data

Grab the first (x, y) sample from the training data to inspect the shape and
data type of an image, and use the class labels to see the FashionMNIST
classes shared by the training and validation data.

In [ ]:
# Take the first sample from the training data.
x_sample, y_sample = train_dataset[0]

# Shape and data type of the x (image) sample.
print(f"x sample shape: {tuple(x_sample.shape)}")
print(f"x sample dtype: {x_sample.dtype}")

# The y sample is a class index; the class names live on the underlying
# FashionMNIST dataset. train_dataset and val_dataset are both Subsets of the
# same train/validation dataset, so they share the same classes.
train_classes = train_dataset.dataset.classes
val_classes = val_dataset.dataset.classes

print(f"\ny sample label index: {y_sample} -> {train_classes[y_sample]}")
print(f"\nTraining classes:   {train_classes}")
print(f"Validation classes: {val_classes}")

## 4. Build the image classifier

Define an `ImageClassifier` as a fully-connected (MLP) network. The constructor
takes the flattened `input_size` (28 x 28 = 784), a list of `hidden_sizes`, and
the number of output `num_classes` (10), and builds an `nn.Sequential` model
from those inputs. Then initialize a model and set up the loss function.

In [ ]:
import torch.nn as nn


class ImageClassifier(nn.Module):
    """A simple fully-connected (MLP) image classifier.

    Args:
        input_size:   Number of input features per image after flattening
                      (e.g. 28 * 28 = 784 for FashionMNIST).
        hidden_sizes: List of hidden-layer widths. One Linear + ReLU block is
                      created for each entry.
        num_classes:  Number of output classes (10 for FashionMNIST).
    """

    def __init__(self, input_size, hidden_sizes, num_classes):
        super().__init__()

        # Build the layers for the Sequential model. Flatten first so the model
        # accepts image tensors of shape (batch, 1, 28, 28) directly.
        layers = [nn.Flatten()]
        in_features = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            in_features = hidden_size
        # Final layer maps to the class logits (no activation; CrossEntropyLoss
        # applies softmax internally).
        layers.append(nn.Linear(in_features, num_classes))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


# Inputs for FashionMNIST: 1 x 28 x 28 images flattened to 784, and 10 classes.
input_size = 1 * 28 * 28
hidden_sizes = [512, 256]
num_classes = 10

# Initialize the model.
model = ImageClassifier(input_size, hidden_sizes, num_classes)

# Loss function.
loss_fn = nn.CrossEntropyLoss()

print(model)
print(f"\nLoss function: {loss_fn}")